In [1]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
output_filename = "/home/tong/recordings/PLYs/reef.ply" 
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# Load and preprocess images from a folder or list of paths

# images = ["/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106044.png"]
# ...

# 1. Provide a list of multiple image paths
images = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104520.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104525.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104530.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104535.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output1047540.png",

]

views = load_images(images)

# Run inference (this will process all images in the list)
predictions1 = model.infer(
    views,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





geometries1 = []
camera_positions = []
PLY = o3d.geometry.PointCloud()

# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
# geometries1.append(world_frame)

last_pose = None
for i, pred1 in enumerate(predictions1):
    
    points_cam = pred1["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred1["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd1 = o3d.geometry.PointCloud()
    pcd1.points = o3d.utility.Vector3dVector(points_cam)
    pcd1.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred1["camera_poses"].squeeze().cpu().numpy()
    
    
    pcd1.transform(camera_pose)

    camera_center = camera_pose[:3, 3]
    camera_positions.append(camera_center)
    
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    
    geometries1.append(pcd1)
    geometries1.append(camera_frame)

    PLY+=pcd1

    if i == len(predictions1) - 1:
        last_pose = pred1["camera_poses"].squeeze().cpu().numpy()  # keep batch dim
         

if len(camera_positions) > 1:
    line_points = o3d.utility.Vector3dVector(camera_positions)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    camera_path.paint_uniform_color([1, 0, 0])
    geometries1.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

for geometry in geometries1:
    geometry.transform(transform_matrix)

PLY.transform(transform_matrix)
# 4. Display all the geometries together in one window
# print(f"Displaying combined scene with {len(predictions1)} point clouds...")
# o3d.visualization.draw_geometries(geometries1)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


PointCloud with 812224 points.

In [3]:

o3d.io.write_point_cloud(output_filename, PLY)

True

In [2]:
o3d.visualization.draw_geometries(geometries1)

In [3]:
print(views[0]['img'].shape)

torch.Size([1, 3, 392, 518])


In [2]:
last_pose

array([[ 0.9993263 , -0.016044  ,  0.03300931, -0.6793342 ],
       [ 0.01432533,  0.998562  ,  0.05165954,  0.7274399 ],
       [-0.03379066, -0.05115186,  0.99811906,  1.6686119 ],
       [ 0.        ,  0.        ,  0.        ,  1.        ]],
      dtype=float32)

In [ ]:
# from PIL import Image
# import numpy as np
# import torch
# from mapanything.models import MapAnything
# device = "cuda" if torch.cuda.is_available() else "cpu"
batch2 = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104820.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104880.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104940.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105000.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105060.png",

]

# last_pose = np.array([
#     [ 0.9993263 , -0.016044  ,  0.03300931, -0.6793342 ],
#     [ 0.01432533,  0.998562  ,  0.05165954,  0.7274399 ],
#     [-0.03379066, -0.05115186,  0.99811906,  1.6686119 ],
#     [ 0.        ,  0.        ,  0.        ,  1.        ]
# ])
# view2_1 = {
#     "img": np.array(Image.open(batch2[0]).convert("RGB")),
#     'camera_poses': last_pose,
# }


# views2 = []
# views2.append(view2_1)

# for i, rgb in enumerate(batch2):
#     if i > 0:
#         views2.append({"img":np.array(Image.open(batch2[i]).convert("RGB"))})


In [ ]:
# view2_1['img'].shape

(300, 400, 3)

In [20]:
# from mapanything.utils.image import preprocess_inputs
# processed_views2 = preprocess_inputs(views2)

views2 = load_images(batch2)

In [21]:
# model = MapAnything.from_pretrained("facebook/map-anything").to(device)
# Run inference with any combination of inputs
predictions2 = model.infer(
    views2,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    # Control which inputs to use/ignore
    # By default, all inputs are used when provided
    # If is_metric_scale flag is not provided, all inputs are assumed to be in metric scale
)

In [22]:
geometries2 = []
camera_positions2 = []
last2_pose = None
for i, pred2 in enumerate(predictions2):



    points2 = pred2["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors2 = pred2["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd2 = o3d.geometry.PointCloud()
    pcd2.points = o3d.utility.Vector3dVector(points2)
    pcd2.colors = o3d.utility.Vector3dVector(colors2)
    camera_pose2 = pred2["camera_poses"].squeeze().cpu().numpy()

    T2 = last_pose@camera_pose2

    pcd2.transform(T2)
    
    

    
    # camera_pose2 = camera_pose2@last_pose
    
    # pcd1.transform(camera_pose)

    camera_center2 = T2[:3, 3]
    camera_positions2.append(camera_center2)
    
    
    camera_frame2 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame2.transform(T2)
    
    
    
    geometries2.append(pcd2)
    geometries2.append(camera_frame2)

    if i == len(predictions2) - 1:
        last2_pose = T2

   

# if len(camera_positions2) > 1:
#     # Define the points for the line set
#     line_points2 = o3d.utility.Vector3dVector(camera_positions2)
#     # Define which points to connect (0->1, 1->2, etc.)
#     line_indices2 = [[i, i + 1] for i in range(len(camera_positions2) - 1)]
#     lines2 = o3d.utility.Vector2iVector(line_indices2)
    
#     # Create the LineSet object
#     camera_path2 = o3d.geometry.LineSet(points=line_points2, lines=lines2)
    
#     # Set the color of the path to red
#     camera_path2.paint_uniform_color([1, 0, 0])
    
#     # Add the path to our list of things to draw
#     geometries2.append(camera_path2)


for geometry in geometries2:
    geometry.transform(transform_matrix)
    

In [23]:
o3d.visualization.draw_geometries(geometries2)

In [24]:
combined_geometries2 = geometries1 + geometries2


In [25]:
o3d.visualization.draw_geometries(
    combined_geometries2
)

In [10]:
last2_pose

array([[ 0.82917318, -0.03921448, -0.55761469, -0.75030004],
       [-0.06272862,  0.98470802, -0.16252747, -1.95885041],
       [ 0.55546111,  0.16974181,  0.8140336 ,  3.38323972],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [26]:
# from PIL import Image
# import numpy as np
# import torch
# from mapanything.models import MapAnything
# device = "cuda" if torch.cuda.is_available() else "cpu"
batch3 = [
   
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105120.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105180.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105240.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105300.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105360.png"

]



# view3_1 = {
#     "img": np.array(Image.open(batch3[0]).convert("RGB")),
#     'camera_poses': last2_pose,
# }


# views3 = []
# views3.append(view3_1)

# for i, rgb in enumerate(batch3):
#     if i > 0:
#         views3.append({"img":np.array(Image.open(batch3[i]).convert("RGB"))})


In [27]:
# processed_views3 = preprocess_inputs(views3)
# model = MapAnything.from_pretrained("facebook/map-anything").to(device)
# Run inference with any combination of inputs
views3 = load_images(batch3)
predictions3 = model.infer(
    views3,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    # Control which inputs to use/ignore
    # By default, all inputs are used when provided
    # If is_metric_scale flag is not provided, all inputs are assumed to be in metric scale
    # ignore_calibration_inputs=False,
    # ignore_depth_inputs=False,
    # ignore_pose_inputs=False,
    # ignore_depth_scale_inputs=False,
    # ignore_pose_scale_inputs=False,
)

In [28]:
geometries3 = []
camera_positions3 = []
last3_pose = None
for i, pred3 in enumerate(predictions3):
    
    points3 = pred3["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors3 = pred3["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd3 = o3d.geometry.PointCloud()
    pcd3.points = o3d.utility.Vector3dVector(points3)
    pcd3.colors = o3d.utility.Vector3dVector(colors3)
    camera_pose3 = pred3["camera_poses"].squeeze().cpu().numpy()
    
    T3 = last2_pose@camera_pose3
    
    pcd3.transform(T3)

    # camera_center3 = camera_pose3[:3, 3]
    # camera_positions3.append(camera_center3)
    
    
    camera_frame3 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame3.transform(T3)
    
    
    
    geometries3.append(pcd3)
    geometries3.append(camera_frame3)

    if i == len(predictions3) - 1:
        last3_pose = T3

   

# if len(camera_positions3) > 1:
#     # Define the points for the line set
#     line_points3 = o3d.utility.Vector3dVector(camera_positions3)
#     # Define which points to connect (0->1, 1->2, etc.)
#     line_indices3 = [[i, i + 1] for i in range(len(camera_positions3) - 1)]
#     lines3 = o3d.utility.Vector2iVector(line_indices3)
    
#     # Create the LineSet object
#     camera_path3 = o3d.geometry.LineSet(points=line_points3, lines=lines3)
    
#     # Set the color of the path to red
#     camera_path3.paint_uniform_color([1, 0, 0])
    
#     # Add the path to our list of things to draw
#     geometries3.append(camera_path3)


for geometry in geometries3:
    geometry.transform(transform_matrix)
    

In [29]:
o3d.visualization.draw_geometries(geometries3)

In [30]:
combined_geometries3 = geometries1 + geometries2 + geometries3

In [31]:
o3d.visualization.draw_geometries(combined_geometries3)

In [52]:
last3_pose

array([[ 0.96541922,  0.11659607,  0.23317629, 10.20787562],
       [-0.06284766,  0.97212408, -0.22588706, -3.47038843],
       [-0.25301382,  0.20342113,  0.94583502, -3.79456264],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [20]:
# from PIL import Image
# import numpy as np
# import torch
# from mapanything.models import MapAnything
# device = "cuda" if torch.cuda.is_available() else "cpu"
batch4 = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106143.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106153.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106163.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106173.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106183.png"

]


views4 = load_images(batch4)
predictions4 = model.infer(
    views4,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)

geometries4 = []
camera_positions4 = []
poses4 = []
last4_pose = None
for i, pred4 in enumerate(predictions4):
    
    points4 = pred4["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors4 = pred4["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd4 = o3d.geometry.PointCloud()
    pcd4.points = o3d.utility.Vector3dVector(points4)
    pcd4.colors = o3d.utility.Vector3dVector(colors4)
    camera_pose4 = pred4["camera_poses"].squeeze().cpu().numpy()

    T4 = camera_pose4

    pcd4.transform(T4)
     
    
    
    
    # camera_pose4 = camera_pose4@last3_pose

    # poses4.append(camera_pose4)
    # pcd4.transform(camera_pose4)
    # # pcd1.transform(camera_pose)

    camera_center4 = camera_pose4[:3, 3]
    camera_positions4.append(camera_center4)
    
    
    camera_frame4 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame4.transform(T4)
    
    
    
    geometries4.append(pcd4)
    geometries4.append(camera_frame4)

    if i == len(predictions4) - 1:
        last4_pose = T4



   

if len(camera_positions4) > 1:
    # Define the points for the line set
    line_points4 = o3d.utility.Vector3dVector(camera_positions4)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices4 = [[i, i + 1] for i in range(len(camera_positions4) - 1)]
    lines4 = o3d.utility.Vector2iVector(line_indices4)
    
    # Create the LineSet object
    camera_path4 = o3d.geometry.LineSet(points=line_points4, lines=lines4)
    
    # Set the color of the path to red
    camera_path4.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries4.append(camera_path4)



# T_pred0 = np.asarray(poses4[0], dtype=np.float64)   # model's first cam->world
# T_ref   = np.asarray(last3_pose, dtype=np.float64)


# T_delta = T_ref @ np.linalg.inv(T_pred0)
# poses4 = [T_delta @ np.asarray(P, dtype=np.float64) for P in poses4]


for g in geometries4:
    # g.transform(T_delta)
    g.transform(transform_matrix)


# for geometry in geometries4:
#     geometry.transform(transform_matrix)
    

In [21]:
o3d.visualization.draw_geometries(geometries4)

In [40]:
combined_geometries4 = geometries1+geometries2+geometries3 + geometries4

In [41]:
o3d.visualization.draw_geometries(combined_geometries4)

In [13]:
batch5 = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105000.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105010.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105020.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105030.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105040.png"

]

views5 = load_images(batch5)
# model = MapAnything.from_pretrained("facebook/map-anything").to(device)
# Run inference with any combination of inputs
predictions5 = model.infer(
    views5,                  # Any combination of input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=False,                  # Apply masking to dense geometry outputs
    mask_edges=False,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)

geometries5 = []
camera_positions5 = []

last5_pose = None
for i, pred5 in enumerate(predictions5):
    
    points5 = pred5["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors5 = pred5["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd5 = o3d.geometry.PointCloud()
    pcd5.points = o3d.utility.Vector3dVector(points5)
    pcd5.colors = o3d.utility.Vector3dVector(colors5)
    camera_pose5 = pred5["camera_poses"].squeeze().cpu().numpy()

    T5 = camera_pose5

    pcd5.transform(T5)
     
    
    
    
    # camera_pose4 = camera_pose4@last3_pose

    # poses4.append(camera_pose4)
    # pcd4.transform(camera_pose4)
    # # pcd1.transform(camera_pose)

    camera_center5 = camera_pose5[:3, 3]
    camera_positions5.append(camera_center5)
    
    
    camera_frame5 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame5.transform(T5)
    
    
    
    geometries5.append(pcd5)
    geometries5.append(camera_frame5)

    if i == len(predictions5) - 1:
        last5_pose = T5



   

if len(camera_positions5) > 1:
    # Define the points for the line set
    line_points5 = o3d.utility.Vector3dVector(camera_positions5)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices5 = [[i, i + 1] for i in range(len(camera_positions5) - 1)]
    lines5 = o3d.utility.Vector2iVector(line_indices5)
    
    # Create the LineSet object
    camera_path5 = o3d.geometry.LineSet(points=line_points5, lines=lines5)
    
    # Set the color of the path to red
    camera_path5.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries5.append(camera_path5)



# T_pred0 = np.asarray(poses4[0], dtype=np.float64)   # model's first cam->world
# T_ref   = np.asarray(last3_pose, dtype=np.float64)


# T_delta = T_ref @ np.linalg.inv(T_pred0)
# poses4 = [T_delta @ np.asarray(P, dtype=np.float64) for P in poses4]


for g in geometries5:
    # g.transform(T_delta)
    g.transform(transform_matrix)


# for geometry in geometries4:
#     geometry.transform(transform_matrix)
    

In [14]:
o3d.visualization.draw_geometries(geometries5)

In [45]:
combined_geometries5 = combined_geometries4 + geometries5

In [46]:
o3d.visualization.draw_geometries(combined_geometries5)

In [5]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
import open3d as o3d
import numpy as np
import glob
import time

# --- Configuration ---
IMAGE_DIR = (
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/"
    "Software/scrippsDivesWithDepth/depth3/linearPNG/"
)
BATCH_SIZE = 5  # Your GPU can handle 5 images at a time
KEYFRAME_STRIDE = 60
FINAL_TRANSFORM = np.array([
    [1,  0,  0,  0],
    [0, -1,  0,  0],
    [0,  0, -1,  0],
    [0,  0,  0,  1]
])

# --- Initial Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)
all_image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))
keyframes_to_process = all_image_files[::KEYFRAME_STRIDE]
num_batches = int(np.ceil(len(keyframes_to_process) / BATCH_SIZE))
print(f"Found {len(all_image_files)} images. Processing in {num_batches} batches of {BATCH_SIZE}.")

# --- Main Processing Loop ---
geometries_to_draw = []
# This matrix "remembers" where the last batch ended. It starts at the world origin.
global_anchor_pose = np.identity(4)

# Process all image files in batches
for i in range(0, len(all_image_files), BATCH_SIZE):
    batch_paths = keyframes_to_process[i:i + BATCH_SIZE]
    if not batch_paths:
        continue
    start_time = time.time()
    print(f"Processing batch {i//BATCH_SIZE + 1}/{num_batches}...")
    
    views = load_images(batch_paths)
    predictions = model.infer(
        views,
        memory_efficient_inference=True,
        use_amp=True,
        apply_mask=False,
        mask_edges=False,
    )
    
    # Process each prediction within the current batch
    for pred in predictions:
        # 1. Get local points and the local pose from the model
        points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        local_camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # 2. Chain the local pose to the end of the previous batch's pose
        current_global_pose = global_anchor_pose @ local_camera_pose
        
        # 3. Create and transform the point cloud into the global world
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points_cam)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        pcd.transform(current_global_pose)
        
        # 4. Create and transform the camera frame
        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0)
        camera_frame.transform(current_global_pose)
        
        # 5. Add geometries and save camera position for the path
        geometries_to_draw.append(pcd)
        geometries_to_draw.append(camera_frame)
    
    # 6. IMPORTANT: Update the anchor to be the pose of the LAST camera in this batch
    # This becomes the starting point for the *next* batch.
    global_anchor_pose = current_global_pose
    
    # Clean up memory for the next batch
    del predictions, views
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    end_time = time.time()
    duration = end_time - start_time
    print(f"--- Batch finished in {duration:.2f} seconds ---")

# --- Final Visualization ---

# # Add the camera path
# if len(camera_positions) > 1:
#     line_path = o3d.geometry.LineSet(
#         points=o3d.utility.Vector3dVector(camera_positions),
#         lines=o3d.utility.Vector2iVector([[i, i + 1] for i in range(len(camera_positions) - 1)])
#     )
#     line_path.paint_uniform_color([1, 0, 0])
#     geometries_to_draw.append(line_path)

# Add the world frame
# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0)
# geometries_to_draw.append(world_frame)

# Apply the final viewing transformation to all objects
for geometry in geometries_to_draw:
    geometry.transform(FINAL_TRANSFORM)



Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Found 1860 images. Processing in 7 batches of 5.
Processing batch 1/7...
--- Batch finished in 3.41 seconds ---
Processing batch 2/7...
--- Batch finished in 3.36 seconds ---
Processing batch 3/7...
--- Batch finished in 3.33 seconds ---
Processing batch 4/7...
--- Batch finished in 3.28 seconds ---
Processing batch 5/7...
--- Batch finished in 3.30 seconds ---
Processing batch 6/7...
--- Batch finished in 3.33 seconds ---
Processing batch 7/7...
--- Batch finished in 0.61 seconds ---


In [6]:
o3d.visualization.draw_geometries(geometries_to_draw)

In [1]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
import open3d as o3d
import numpy as np
import glob
import time

# --- Configuration ---
IMAGE_DIR = (
    '/home/tong/recordings/maritime1' 

)
BATCH_SIZE = 5  # Your GPU can handle 5 images at a time
KEYFRAME_STRIDE = 1
FINAL_TRANSFORM = np.array([
    [1,  0,  0,  0],
    [0, -1,  0,  0],
    [0,  0, -1,  0],
    [0,  0,  0,  1]
])

# --- Initial Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)
all_image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))
keyframes_to_process = all_image_files[::KEYFRAME_STRIDE]
num_batches = int(np.ceil(len(keyframes_to_process) / BATCH_SIZE))
print(f"Found {len(all_image_files)} images. Processing in {num_batches} batches of {BATCH_SIZE}.")

# --- Main Processing Loop ---
maritime1 = []
# This matrix "remembers" where the last batch ended. It starts at the world origin.
global_anchor_pose = np.identity(4)

# Process all image files in batches
for i in range(0, len(all_image_files), BATCH_SIZE):
    batch_paths = keyframes_to_process[i:i + BATCH_SIZE]
    if not batch_paths:
        continue
    start_time = time.time()
    print(f"Processing batch {i//BATCH_SIZE + 1}/{num_batches}...")
    
    views = load_images(batch_paths)
    predictions = model.infer(
        views,
        memory_efficient_inference=True,
        use_amp=True,
        apply_mask=False,
        mask_edges=False,
    )
    
    # Process each prediction within the current batch
    for pred in predictions:
        # 1. Get local points and the local pose from the model
        points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        local_camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # 2. Chain the local pose to the end of the previous batch's pose
        current_global_pose = global_anchor_pose @ local_camera_pose
        
        # 3. Create and transform the point cloud into the global world
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points_cam)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        pcd.transform(current_global_pose)
        
        # 4. Create and transform the camera frame
        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0)
        camera_frame.transform(current_global_pose)
        
        # 5. Add geometries and save camera position for the path
        maritime1.append(pcd)
        maritime1.append(camera_frame)
    
    # 6. IMPORTANT: Update the anchor to be the pose of the LAST camera in this batch
    # This becomes the starting point for the *next* batch.
    global_anchor_pose = current_global_pose
    
    # Clean up memory for the next batch
    del predictions, views
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    end_time = time.time()
    duration = end_time - start_time
    print(f"--- Batch finished in {duration:.2f} seconds ---")

# --- Final Visualization ---

# # Add the camera path
# if len(camera_positions) > 1:
#     line_path = o3d.geometry.LineSet(
#         points=o3d.utility.Vector3dVector(camera_positions),
#         lines=o3d.utility.Vector2iVector([[i, i + 1] for i in range(len(camera_positions) - 1)])
#     )
#     line_path.paint_uniform_color([1, 0, 0])
#     geometries_to_draw.append(line_path)

# Add the world frame
# world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0)
# geometries_to_draw.append(world_frame)

# Apply the final viewing transformation to all objects
for geometry in maritime1:
    geometry.transform(FINAL_TRANSFORM)



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Found 81 images. Processing in 17 batches of 5.
Processing batch 1/17...
--- Batch finished in 3.68 seconds ---
Processing batch 2/17...
--- Batch finished in 3.42 seconds ---
Processing batch 3/17...
--- Batch finished in 3.47 seconds ---
Processing batch 4/17...
--- Batch finished in 3.47 seconds ---
Processing batch 5/17...
--- Batch finished in 3.50 seconds ---
Processing batch 6/17...
--- Batch finished in 3.53 seconds ---
Processing batch 7/17...
--- Batch finished in 3.51 seconds ---
Processing batch 8/17...
--- Batch finished in 3.45 seconds ---
Processing batch 9/17...
--- Batch finished in 3.48 seconds ---
Processing batch 10/17...
--- Batch finished in 3.49 seconds ---
Processing batch 11/17...
--- Batch finished in 3.51 seconds ---
Processing batch 12/17...
--- Batch finished in 3.51 seconds ---
Processing batch 13/17...
--- Batch finished in 3.52 seconds ---
Processing batch 14/17...
--- Batch finished in 3.49 seconds ---
Processing batch 15/17...
--- Batch finished in 3.5

In [2]:

o3d.visualization.draw_geometries(maritime1)